# Titanic — Target-Encoded Logistic Regression Reproduction

Reproduces the target-encoded Logistic Regression submission.

**Kaggle public ROC-AUC:** `0.78218`

Run this notebook from top to bottom in VS Code. The final verification cell compares the regenerated CSV with the exact submitted reference CSV.

## 1. Imports and exact model logic

The functions below are reconstructed from the original Work source. Comments explain why each step exists. **ROC-AUC** means *Area Under the Receiver Operating Characteristic Curve* and rewards correct ranking of survivors above non-survivors.

In [ ]:

import re, math
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import roc_auc_score

SEED = 20261029
ALPHA = 5.0


def title_from_name(value):
    """Extract Mr/Miss/Mrs/Master; group all other titles as Rare."""
    match = re.search(r",\s*([^.]*)\.", str(value))
    title = match.group(1).strip() if match else "Unknown"
    common = {"Mr", "Miss", "Mrs", "Master"}
    return title if title in common else "Rare"


def engineered_with_reference(raw, reference):
    """Create the exact predictor-only engineered features used by Work."""
    df = raw.copy()
    ref = reference.copy()
    for frame in (df, ref):
        frame["Surname"] = frame["Name"].astype(str).str.split(",").str[0].str.strip()
        frame["FamilySize"] = frame["SibSp"].fillna(0) + frame["Parch"].fillna(0) + 1

    ticket_counts = ref["Ticket"].astype(str).value_counts(dropna=False)
    family_keys = ref["Surname"].astype(str) + "|" + ref["FamilySize"].astype(str)
    family_counts = family_keys.value_counts(dropna=False)

    df["Title"] = df["Name"].map(title_from_name)
    df["FamilySize"] = df["SibSp"].fillna(0) + df["Parch"].fillna(0) + 1
    df["IsAlone"] = (df["FamilySize"] == 1).astype(int)
    df["TicketGroupSize"] = df["Ticket"].astype(str).map(ticket_counts).fillna(1).astype(float)
    df["FarePerPerson"] = df["Fare"] / df["TicketGroupSize"]
    df["CabinKnown"] = df["Cabin"].notna().astype(int)
    df["Deck"] = df["Cabin"].fillna("U").astype(str).str[0]
    df["AgeMissing"] = df["Age"].isna().astype(int)
    df["FareMissing"] = df["Fare"].isna().astype(int)
    df["Surname"] = df["Name"].astype(str).str.split(",").str[0].str.strip()
    df["FamilyGroupSize"] = (
        df["Surname"].astype(str) + "|" + df["FamilySize"].astype(str)
    ).map(family_counts).fillna(1).astype(float)
    df["Child"] = (df["Age"].fillna(30) < 14).astype(int)
    df["Mother"] = (
        (df["Sex"] == "female")
        & (df["Parch"].fillna(0) > 0)
        & (df["Age"].fillna(30) > 18)
        & (df["Title"] != "Miss")
    ).astype(int)
    df["SexPclass"] = df["Sex"].astype(str) + "_" + df["Pclass"].astype(str)
    df["TitlePclass"] = df["Title"].astype(str) + "_" + df["Pclass"].astype(str)

    return df[[
        "Sex", "Pclass", "Age", "SibSp", "Parch", "Fare", "Embarked",
        "CabinKnown", "AgeMissing", "FareMissing", "Title", "FamilySize",
        "IsAlone", "TicketGroupSize", "FarePerPerson", "Deck",
        "FamilyGroupSize", "Child", "Mother", "SexPclass", "TitlePclass",
    ]]


def target_encode(train_raw, y_train, apply_raw, key_name, alpha=ALPHA):
    """Fold-safe/leave-one-out target encoding used for Surname and Ticket."""
    train_keys = (
        train_raw["Name"].astype(str).str.split(",").str[0].str.strip()
        if key_name == "Surname" else train_raw["Ticket"].astype(str)
    )
    apply_keys = (
        apply_raw["Name"].astype(str).str.split(",").str[0].str.strip()
        if key_name == "Surname" else apply_raw["Ticket"].astype(str)
    )

    sums = defaultdict(float)
    counts = defaultdict(int)
    for key, label in zip(train_keys, y_train):
        sums[key] += float(label)
        counts[key] += 1

    global_mean = float(np.mean(y_train))
    train_values = np.empty(len(train_raw), dtype=float)
    for i, (key, label) in enumerate(zip(train_keys, y_train)):
        train_values[i] = (
            sums[key] - float(label) + alpha * global_mean
        ) / (max(counts[key] - 1, 0) + alpha)

    apply_values = np.array([
        (sums[key] + alpha * global_mean) / (counts[key] + alpha)
        if counts[key] else global_mean
        for key in apply_keys
    ])
    return train_values, apply_values


def add_target_features(train_features, apply_features, train_raw, y_train, apply_raw):
    left = train_features.copy()
    right = apply_features.copy()
    for key_name in ("Surname", "Ticket"):
        left_values, right_values = target_encode(train_raw, y_train, apply_raw, key_name)
        left[f"{key_name}TargetMean"] = left_values
        right[f"{key_name}TargetMean"] = right_values
    return left, right


class MatrixEncoder:
    """Exact lightweight encoder used in the Work pipeline: one-hot + median impute + standardize."""
    def __init__(self, categorical, numeric):
        self.categorical = categorical
        self.numeric = numeric
        self.categories = {}
        self.medians = {}
        self.means = None
        self.scales = None

    def fit(self, df):
        for col in self.categorical:
            values = df[col].fillna("__MISSING__").astype(str)
            cats = sorted(set(values.tolist()))
            if "__UNK__" not in cats:
                cats.append("__UNK__")
            self.categories[col] = cats
        for col in self.numeric:
            values = pd.to_numeric(df[col], errors="coerce")
            median = values.median()
            self.medians[col] = float(median) if not math.isnan(float(median)) else 0.0
        raw = self._raw_matrix(df)
        self.means = raw.mean(axis=0)
        self.scales = raw.std(axis=0)
        self.scales[self.scales < 1e-9] = 1.0
        return self

    def _raw_matrix(self, df):
        parts = []
        for col in self.categorical:
            values = df[col].fillna("__MISSING__").astype(str).tolist()
            cats = self.categories[col]
            index = {value: i for i, value in enumerate(cats)}
            matrix = np.zeros((len(values), len(cats)), dtype=float)
            for row, value in enumerate(values):
                matrix[row, index.get(value, index["__UNK__"])] = 1.0
            parts.append(matrix)
        for col in self.numeric:
            values = pd.to_numeric(df[col], errors="coerce").fillna(self.medians[col]).to_numpy(dtype=float)
            parts.append(values.reshape(-1, 1))
        return np.concatenate(parts, axis=1) if parts else np.zeros((len(df), 0))

    def transform(self, df):
        raw = self._raw_matrix(df)
        return (raw - self.means) / self.scales


def encode(train_features, apply_features):
    categorical = [c for c in train_features.columns if not pd.api.types.is_numeric_dtype(train_features[c])]
    numeric = [c for c in train_features.columns if c not in categorical]
    encoder = MatrixEncoder(categorical, numeric).fit(train_features)
    return encoder.transform(train_features), encoder.transform(apply_features)


def logistic_model():
    return LogisticRegression(C=12.5, max_iter=3000, solver="lbfgs", random_state=SEED)


def random_forest_model():
    return RandomForestClassifier(
        n_estimators=350,
        min_samples_leaf=2,
        max_features="sqrt",
        random_state=SEED,
        n_jobs=1,
    )


## 2. Load the competition data

`train.csv` contains the target `Survived`; `test.csv` contains the passengers to predict. `gender_submission.csv` supplies the required Kaggle output IDs and column order.

In [ ]:
from pathlib import Path
import hashlib
from urllib.request import urlretrieve

ROOT = Path.cwd()
DATA = ROOT / "data"
REFERENCE = ROOT / "reference"
OUTPUT = ROOT / "output"
DATA.mkdir(exist_ok=True)
OUTPUT.mkdir(exist_ok=True)

DATA_URLS = {'train.csv': 'https://raw.githubusercontent.com/minsuk-heo/kaggle-titanic/master/input/train.csv', 'test.csv': 'https://raw.githubusercontent.com/minsuk-heo/kaggle-titanic/master/input/test.csv', 'gender_submission.csv': 'https://raw.githubusercontent.com/minsuk-heo/kaggle-titanic/master/input/gender_submission.csv'}
EXPECTED_SHA256 = {'train.csv': '7d118fef8b6ccf7f81111877bc388536f7b1e498a655e3d649d19aaa010e9f6f', 'test.csv': '56023b9948236f3c7a1c9448fcf418b283e109ef177fa8c7e069158dd7dd52b2', 'gender_submission.csv': '7a262a7f4cb807f53911dccc84257ed3976b21b7b50ffc468b58446bff8bb6a4'}

def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

for filename, url in DATA_URLS.items():
    path = DATA / filename
    if not path.exists() or sha256_file(path) != EXPECTED_SHA256[filename]:
        print(f"Downloading exact Titanic input: {filename}")
        urlretrieve(url, path)
    actual = sha256_file(path)
    assert actual == EXPECTED_SHA256[filename], f"Hash mismatch for {filename}: {actual}"

train = pd.read_csv(DATA / "train.csv")
test = pd.read_csv(DATA / "test.csv")
sample = pd.read_csv(DATA / "gender_submission.csv")

raw_train = train.drop(columns=["Survived"])
raw_test = test.copy()
y = train["Survived"].to_numpy(dtype=int)
all_predictors = pd.concat([raw_train, raw_test], ignore_index=True)

print(f"train: {train.shape}; test: {test.shape}")
print("Input hashes verified: PASS")
print("target counts:", train["Survived"].value_counts().sort_index().to_dict())


## 3. Feature engineering and target encoding

We create passenger-level features such as title, family size, travelling alone, ticket-group size, deck, child/mother flags and interactions. Then we create **SurnameTargetMean** and **TicketTargetMean** from training labels using leave-one-out smoothing (`alpha=5`) for training rows.

In [ ]:

# Predictor-only engineered features use train + test predictor rows for group counts.
# No test Survived labels exist or are used.
feat_full = engineered_with_reference(raw_train, all_predictors)
feat_test = engineered_with_reference(raw_test, all_predictors)

# Add SurnameTargetMean and TicketTargetMean.
# Training values use leave-one-out target encoding so a row does not directly encode its own label.
target_full, target_test = add_target_features(feat_full, feat_test, raw_train, y, raw_test)
tx_full, tx_test = encode(target_full, target_test)

print("Encoded target-logistic matrix:", tx_full.shape, tx_test.shape)


## 4. Train the final Logistic Regression and create the Kaggle CSV

The exact model uses `C=12.5`, `lbfgs`, `max_iter=3000`, and random seed `20261029`. `predict_proba(... )[:,1]` gives the probability of `Survived=1`, which is what ROC-AUC needs.

In [ ]:

model = logistic_model()
model.fit(tx_full, y)
logistic_probability = model.predict_proba(tx_test)[:, 1]

submission = sample[["PassengerId"]].copy()
submission["Survived"] = logistic_probability
output_path = OUTPUT / "final_target_logistic_candidate.csv"
submission.to_csv(output_path, index=False)

print("Created:", output_path)
print("Prediction range:", float(logistic_probability.min()), "to", float(logistic_probability.max()))
submission.head()


## 5. Reproduce the local validation score

This is **Repeated Stratified K-Fold Cross-Validation**: 5 folds repeated 3 times = 15 validation folds. It estimates model ranking performance on held-out training passengers.

In [ ]:

# Reproduce the repeated 5-fold x 3 validation used in the final comparison.
folds = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=SEED)
scores = []
for train_idx, valid_idx in folds.split(np.zeros(len(y)), y):
    raw_tr = raw_train.iloc[train_idx]
    raw_va = raw_train.iloc[valid_idx]
    feat_tr = engineered_with_reference(raw_tr, all_predictors)
    feat_va = engineered_with_reference(raw_va, all_predictors)
    target_tr, target_va = add_target_features(feat_tr, feat_va, raw_tr, y[train_idx], raw_va)
    x_tr, x_va = encode(target_tr, target_va)
    m = logistic_model().fit(x_tr, y[train_idx])
    pred = m.predict_proba(x_va)[:, 1]
    scores.append(roc_auc_score(y[valid_idx], pred))

print(f"Repeated CV ROC-AUC mean: {np.mean(scores):.6f}")
print(f"Repeated CV ROC-AUC SD:   {np.std(scores, ddof=1):.6f}")
print("Original Work result was approximately 0.882810 ± 0.026435.")


## 6. Verify against the exact submitted reference

This is the key proof: the regenerated `PassengerId` sequence must match exactly, and prediction differences should be zero or numerically negligible.

In [ ]:

reference = pd.read_csv(REFERENCE / "final_target_logistic_candidate.csv")
created = pd.read_csv(OUTPUT / "final_target_logistic_candidate.csv")

assert created.columns.tolist() == ["PassengerId", "Survived"]
assert len(created) == 418
assert created["PassengerId"].equals(reference["PassengerId"])

max_abs_diff = float(np.max(np.abs(created["Survived"].to_numpy() - reference["Survived"].to_numpy())))
print("Maximum absolute prediction difference vs submitted reference:", max_abs_diff)
print("Exact CSV values reproduced:", bool(max_abs_diff == 0.0))
print("Numerically identical to 1e-12:", bool(max_abs_diff <= 1e-12))


## How to explain this submission

- **Target:** predict probability that `Survived = 1`.
- **Metric:** ROC-AUC (Area Under the Receiver Operating Characteristic Curve), so continuous probabilities are submitted instead of hard 0/1 labels.
- **Feature engineering:** title, family size, travelling alone, ticket/family group sizes, fare per person, cabin/deck, missingness flags, child/mother flags and class interactions.
- **Target encoding:** surname and ticket survival information is smoothed and leave-one-out for training rows to reduce direct self-leakage.
- **Validation caution:** ordinary stratified folds were optimistic because related passengers can appear in both training and validation folds. Later group-aware checks showed a lower score.
- **Why the blend works:** the two models make different ranking errors; averaging probabilities can improve the final ranking.